# IAD Pipeline
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

## 1. Environment Setup
Configure database URI and API keys.

In [ ]:
import os
import sys
import warnings

# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'


sys.path.append("..")

# Now safe to import
import wandb
import logging
from pathlib import Path
from src.manager import AnomalyDetectionManager as ADM

wandb.login()

## 2. Configuration
Set your run parameters here before executing the pipeline.

In [ ]:

train       = True
evaluate    = True
datasetDir  = Path("../datasets/")
configDir   = Path("../configs/")
outputPath  = Path("../results/")
product     = Path("Products/cable.yaml")
logger = logging.getLogger("logger")
manager = ADM.loadProduct(productConfig=Path(configDir/product), outputPath=outputPath, configDir=configDir, datasetDir=datasetDir)

## 4.1 Inspect Dataset

In [ ]:
manager.launchSession()

## 5. Training

In [ ]:

warnings.filterwarnings("ignore", category=FutureWarning, module="timm.models.layers")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="openvino.runtime")
if train:
    manager.train(manager.trainingPath, tiling=manager.isTilingSetup)

In [ ]:
manager.launchSession()

## 6. Evaluation & Prediction

In [ ]:
if evaluate:
    if manager.ckptPath is not None:
        if not manager.isTilingSetup:
            manager.loadCheckpoint(manager.ckptPath, f"{manager.modelName}")
        manager.eval(configDir / "Trainer"/ "Evaluation.yaml", tiling=manager.isTilingSetup)
        manager.launchSession()
    else:
        print("No checkpoint found — skipping evaluation.")
    

## 7. Evaluate on single unknown Image

In [ ]:
# ckptPath = manager.ckptPath.parent.resolve()
# print(ckptPath)

In [ ]:

# # ckptPath = manager.ckptPath.parent.resolve()
# # print(ckptPath)
# manager.loadDatasetFromDisk(datasetPath=datasetDir / "MVTecADShortPred", datasetName="cablePred36", split=("pred",), overwrite=True)
# # manager.selectCategory("cable")

# if manager.ckptPath is not None:
#     if not tiling:
#         manager.loadCheckpoint(manager.ckptPath, f"{modelName}")
#     if tiling:
#         manager.setupTiling(configDir / "TiledEnsemblePred.yaml")
#     manager.predict(config=configDir / "Predict.yaml", tiling=tiling, ckptPath=ckptPath)
#     print(manager.FO_Dataset)
#     manager.launchSession()
#     print(manager.FO_Dataset)
#     manager.launchSession()